# 9/11 Revision

Instead of an external bias, make kappa itself a function of time. The stickiness of a word should naturally decay the longer we've been in that state.

The logic:

1. Immediately after transitioning to `"sure"`, the `kappa` for `"sure"` is very high (we expect to stay there).
2. As time passes within the `"sure"` state, its `kappa` value gradually decreases.
3. This decreasing `kappa` means the probability of transitioning *out* of `"sure"` (to `"or"` or `"time"`) naturally increases over time.

In [ ]:
import numpy as np
from collections import Counter
from scipy.stats import norm

np.random.seed(7)

# STATES, EMIT_STATES, mu, sigma, make_pi_bar, pi_bar
# kappa_from_gamma_rate, sample_gammas, build_T, expected_dwell
# simulate_one, emission_log_probs, compress
# ------------------------
# States and emissions
# ------------------------
STATES = ["START","lei","sure","or","time","STOP"]
EMIT_STATES = {"lei","sure","or","time"}
i = {s:k for k,s in enumerate(STATES)}

# 1D acoustic feature: 'lei' low, 'time' high, 'sure' and 'or' similar
mu = {
    "lei":  -1.0,
    "sure":  0.15,
    "or":    0.05,
    "time":  1.0
}
sigma = {s:0.45 for s in EMIT_STATES}

# ------------------------
# Base transitions (pi_bar)
# ------------------------
def make_pi_bar(uniform_eps=0.0):
    n = len(STATES)
    T = np.zeros((n,n))
    def row_with_primary(from_s, primary):
        row = np.zeros(n)
        used = 0.0
        for to_s, w in primary.items():
            row[i[to_s]] += w; used += w
        leftover = max(0.0, 1.0 - used)
        mask = np.ones(n, dtype=bool)
        for to_s in primary.keys():
            mask[i[to_s]] = False
        mask[i["START"]] = False
        if from_s == "STOP":
            mask[:] = False
            mask[i["STOP"]] = True
        count = mask.sum()
        if count > 0 and leftover > 0:
            row[mask] += leftover / count
        if uniform_eps > 0:
            row += uniform_eps
        row = row / row.sum()
        return row

    T[i["START"]] = row_with_primary("START", {"lei":0.97})
    T[i["lei"]]   = row_with_primary("lei",   {"sure":0.94})
    T[i["sure"]]  = row_with_primary("sure",  {"or":0.55, "time":0.40})
    T[i["or"]]    = row_with_primary("or",    {"time":0.95})
    T[i["time"]]  = row_with_primary("time",  {"STOP":0.96})
    T[i["STOP"]]  = row_with_primary("STOP",  {"STOP":1.0})
    return T

pi_bar = make_pi_bar(uniform_eps=0.0)

# ------------------------
# Stickiness mapping kappa(gamma, rate)
# ------------------------
def kappa_from_gamma_rate(gamma_val, rate, base=1.2, clip=(0.55,0.93)):
    val = 1.0 - np.exp(-base * (gamma_val + 0.25) / (rate + 1e-9))
    return float(np.clip(val, clip[0], clip[1]))

def sample_gammas(seed=None):
    rng = np.random.default_rng(seed)
    g = {}
    for s in STATES:
        if s in {"START","STOP"}: g[s] = 0.0
        else: g[s] = rng.gamma(shape=2.0, scale=1/5.0)
    return g

# Build a *base* T matrix (not used in dynamic decoder, but used for simulation)
def build_T(rate, gammas):
    n = len(STATES)
    T = np.zeros((n,n))
    for s_from in STATES:
        row = pi_bar[i[s_from]].copy()
        if s_from in {"START","STOP"}:
            T[i[s_from]] = row
        else:
            kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            row = (1 - kappa) * row
            row[i[s_from]] += kappa
            T[i[s_from]] = row / row.sum()
    return T

# ------------------------
# Generative simulation
# ------------------------
def simulate_one(rate, gammas, max_frames=1000, rng=None):
    if rng is None: rng = np.random.default_rng()
    T = build_T(rate, gammas)
    s = "START"
    states, emissions = [], []
    frame = 0
    while frame < max_frames:
        s = rng.choice(STATES, p=T[i[s]])
        if s == "STOP": break
        states.append(s)
        if s in EMIT_STATES:
            emissions.append(rng.normal(loc=mu[s], scale=sigma[s]))
        frame += 1
    return states, np.array(emissions).reshape(-1,1)


# New revisions

def dynamic_kappa(t_elapsed, base_kappa, word_expected_duration):
    """
    Calculates a dynamic kappa that decays over time.
    Starts near 1.0 and decays towards a floor value.
    The rate of decay depends on the word's expected duration.
    """
    # A steepness factor for the decay curve.
    decay_rate = 5.0 / max(1.0, word_expected_duration)
    # A reversed logistic function to model the decay.
    # It starts high and decreases smoothly.
    val = base_kappa + (0.99 - base_kappa) / (1 + np.exp(decay_rate * (t_elapsed - word_expected_duration / 2.0)))
    return val

def get_dynamic_logT(rate, gammas, dwell_times):
    """
    Computes the log transition matrix for the current time step,
    given how long we've been in each of the previous states.
    """
    n = len(STATES)
    logT = np.full((n, n), -np.inf)

    # Calculate base expected durations for decay rate reference
    expected_durations = {}
    for s_from in STATES:
        if s_from in EMIT_STATES:
            base_kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            expected_durations[s_from] = 1.0 / (1.0 - base_kappa)
        else:
            expected_durations[s_from] = 1.0

    for s_from_idx in range(n):
        s_from = STATES[s_from_idx]
        row = pi_bar[s_from_idx].copy()

        if s_from in {"START", "STOP"}:
            logT[s_from_idx, :] = np.log(row + 1e-12)
        else:
            # Get the elapsed time for this specific state
            t_elapsed = dwell_times[s_from_idx]

            # Calculate the kappa for *this moment in time*
            base_kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            kappa_t = dynamic_kappa(t_elapsed, base_kappa, expected_durations[s_from])

            # Build the transition row
            row = (1 - kappa_t) * row
            row[s_from_idx] += kappa_t
            logT[s_from_idx, :] = np.log(row / row.sum() + 1e-12)

    return logT

def viterbi_decode_dynamic(a_data, rate, gammas):
    """
    Viterbi decoder that uses time-dependent stickiness.
    """
    T_len = len(a_data)
    n = len(STATES)
    V = np.full((T_len, n), -np.inf) # Log probability of the best path
    B = np.zeros((T_len, n), dtype=int) # Backpointers to previous state
    D = np.zeros((T_len, n), dtype=int) # Dwell time: how long path has been in current state
    logE = emission_log_probs(a_data)

    # Initialization at t=0
    # Probabilities of transitioning from START to each state
    start_row = np.log(pi_bar[i["START"]] + 1e-12)
    for s_idx in range(n):
        if STATES[s_idx] in EMIT_STATES:
            V[0, s_idx] = start_row[s_idx] + logE[0, s_idx]
            D[0, s_idx] = 1 # Just started, so dwell time is 1

    # Main loop for t > 0
    for t in range(1, T_len):
        # For each possible current state j...
        for j in range(n):
            # Calculate the dynamic transition matrix based on dwell times at t-1
            logT_t = get_dynamic_logT(rate, gammas, D[t-1])

            # Calculate probabilities of all paths from previous states k to current state j
            prev_path_probs = V[t-1, :] + logT_t[:, j]

            # Find the best previous state k
            best_prev_k = np.argmax(prev_path_probs)

            # Store the probability and backpointer
            V[t, j] = prev_path_probs[best_prev_k] + logE[t, j]
            B[t, j] = best_prev_k

            # Update the dwell time
            if j == best_prev_k:
                D[t, j] = D[t-1, best_prev_k] + 1 # Continued in same state
            else:
                D[t, j] = 1 # Just transitioned to a new state

    # Backtrack to find the best path
    path = np.zeros(T_len, dtype=int)
    path[-1] = np.argmax(V[-1])
    for t in range(T_len-2, -1, -1):
        path[t] = B[t+1, path[t+1]]

    return [STATES[k] for k in path]

# Helper functions for the experiment
def emission_log_probs(a_data):
    T = len(a_data)
    n = len(STATES)
    logp = np.full((T, n), -np.inf)
    for t in range(T):
        for s in EMIT_STATES:
            logp[t, i[s]] = norm.logpdf(a_data[t,0], loc=mu[s], scale=sigma[s])
    return logp

def compress(seq):
    return [s for i, s in enumerate(seq) if i == 0 or s != seq[i-1]]


# Monte Carlo simulations
def run_revised_experiment(n_trials=1000, rate_slow=4.0, rate_fast=7.0):
    rng = np.random.default_rng(1234)
    gammas = sample_gammas(seed=2025)
    seq_slow = []
    seq_fast = []

    for k in range(n_trials):
        if k % 100 == 0: print(f"Trial {k}/{n_trials}...")
        # Slow: generate and decode with slow rate
        s_states, a = simulate_one(rate_slow, gammas, rng=rng)
        if len(a) < 3: continue
        zhat_slow = viterbi_decode_dynamic(a, rate_slow, gammas)
        seq_slow.append(compress(zhat_slow))

        # Fast: generate and decode with fast rate
        f_states, a2 = simulate_one(rate_fast, gammas, rng=rng)
        if len(a2) < 3: continue
        zhat_fast = viterbi_decode_dynamic(a2, rate_fast, gammas)
        seq_fast.append(compress(zhat_fast))

    def summarize(name, seqs):
        c = Counter(tuple(s) for s in seqs)
        total = sum(c.values())
        print(f"\n{name} (N={total}) top 8:")
        for (s, cnt) in c.most_common(8):
            print(f"  {list(s)!s:<40} | {cnt:4d} | {100*cnt/total:5.1f}%")
        has_or = sum(1 for s in seqs if "or" in s)
        print(f"  -> 'or' present: {has_or}/{total} = {100*has_or/total:4.1f}%")

    print("\n\n=== With DYNAMIC TIME-DEPENDENT Stickiness ===")
    summarize(f"SLOW r={rate_slow}", seq_slow)
    summarize(f"FAST r={rate_fast}", seq_fast)

# Run the new experiment
run_revised_experiment(n_trials=1000)

Trial 0/1000...
Trial 100/1000...
Trial 200/1000...
Trial 300/1000...
Trial 400/1000...
Trial 500/1000...
Trial 600/1000...
Trial 700/1000...
Trial 800/1000...
Trial 900/1000...


=== With DYNAMIC TIME-DEPENDENT Stickiness ===

SLOW r=4.0 (N=971) top 8:
  ['lei', 'sure', 'time']                  |  460 |  47.4%
  ['lei', 'sure', 'or', 'time']            |  254 |  26.2%
  ['lei', 'sure']                          |  221 |  22.8%
  ['or', 'time']                           |    7 |   0.7%
  ['lei', 'time']                          |    7 |   0.7%
  ['lei', 'sure', 'lei', 'sure', 'or', 'time'] |    4 |   0.4%
  ['lei', 'sure', 'lei', 'sure', 'time']   |    3 |   0.3%
  ['time']                                 |    3 |   0.3%
  -> 'or' present: 270/971 = 27.8%

FAST r=7.0 (N=944) top 8:
  ['lei', 'sure', 'time']                  |  438 |  46.4%
  ['lei', 'sure', 'or', 'time']            |  246 |  26.1%
  ['lei', 'sure']                          |  204 |  21.6%
  ['or', 'time']               

# Possible issue: confounding the "Speaker" and the "Listener"?
It changes the speech rate for both the generative process ("the speaker") and the decoding process ("the listener") at the same time.

The Speaker (simulate_one): When you use a fast rate, the speaker simulation naturally produces shorter words and is more likely to generate the "or" state to begin with. The opposite is true for the slow rate.

The Listener (viterbi_decode_dynamic): The listener then uses a corresponding fast or slow "perceptual model" with dynamic stickiness to interpret the signal it receives.

The "listener" is interpreting a signal that was already biased by the speaker. The effect of the kappa in the decoder is being applied to data that is different in each condition. This setup masks the pure perceptual effect we want to isolate.



In [ ]:
import numpy as np
from collections import Counter
from scipy.stats import norm

np.random.seed(7)

# ------------------------
# States and emissions
# ------------------------
STATES = ["START", "lei", "sure", "or", "time", "STOP"]
EMIT_STATES = {"lei", "sure", "or", "time"}
i = {s: k for k, s in enumerate(STATES)}

# 1D acoustic feature: 'lei' low, 'time' high, 'sure' and 'or' similar
mu = {
    "lei":  -1.0,
    "sure":  0.15,
    "or":    0.05,
    "time":  1.0
}
sigma = {s: 0.45 for s in EMIT_STATES}

# ------------------------
# Base transitions (pi_bar)
# ------------------------
def make_pi_bar(uniform_eps=0.0):
    n = len(STATES)
    T = np.zeros((n, n))
    def row_with_primary(from_s, primary):
        row = np.zeros(n)
        used = 0.0
        for to_s, w in primary.items():
            row[i[to_s]] += w
            used += w
        leftover = max(0.0, 1.0 - used)
        mask = np.ones(n, dtype=bool)
        for to_s in primary.keys():
            mask[i[to_s]] = False
        mask[i["START"]] = False
        if from_s == "STOP":
            mask[:] = False
            mask[i["STOP"]] = True
        count = mask.sum()
        if count > 0 and leftover > 0:
            row[mask] += leftover / count
        if uniform_eps > 0:
            row += uniform_eps
        row = row / row.sum()
        return row

    T[i["START"]] = row_with_primary("START", {"lei": 0.97})
    T[i["lei"]]   = row_with_primary("lei",   {"sure": 0.94})
    T[i["sure"]]  = row_with_primary("sure",  {"or": 0.55, "time": 0.40})
    T[i["or"]]    = row_with_primary("or",    {"time": 0.95})
    T[i["time"]]  = row_with_primary("time",  {"STOP": 0.96})
    T[i["STOP"]]  = row_with_primary("STOP",  {"STOP": 1.0})
    return T

pi_bar = make_pi_bar(uniform_eps=0.0)

# ------------------------
# Stickiness mapping kappa(gamma, rate)
# ------------------------
def kappa_from_gamma_rate(gamma_val, rate, base=1.2, clip=(0.55, 0.93)):
    val = 1.0 - np.exp(-base * (gamma_val + 0.25) / (rate + 1e-9))
    return float(np.clip(val, clip[0], clip[1]))

def sample_gammas(seed=None):
    rng = np.random.default_rng(seed)
    g = {}
    for s in STATES:
        if s in {"START", "STOP"}: g[s] = 0.0
        else: g[s] = rng.gamma(shape=2.0, scale=1/5.0)
    return g

def build_T(rate, gammas):
    n = len(STATES)
    T = np.zeros((n, n))
    for s_from in STATES:
        row = pi_bar[i[s_from]].copy()
        if s_from in {"START", "STOP"}:
            T[i[s_from]] = row
        else:
            kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            row = (1 - kappa) * row
            row[i[s_from]] += kappa
            T[i[s_from]] = row / row.sum()
    return T

# ------------------------
# Generative simulation
# ------------------------
def simulate_one(rate, gammas, max_frames=1000, rng=None):
    if rng is None: rng = np.random.default_rng()
    T = build_T(rate, gammas)
    s = "START"
    states, emissions = [], []
    frame = 0
    while frame < max_frames:
        s = rng.choice(STATES, p=T[i[s]])
        if s == "STOP": break
        states.append(s)
        if s in EMIT_STATES:
            emissions.append(rng.normal(loc=mu[s], scale=sigma[s]))
        frame += 1
    return states, np.array(emissions).reshape(-1, 1)

# =================================================================
# == NEW DYNAMIC STICKINESS & VITERBI DECODER ==
# =================================================================

def dynamic_kappa_refined(t_elapsed, base_kappa, word_expected_duration, rate):
    """
    A refined kappa function where the decay rate is explicitly tied to speech rate.
    """
    urgency_factor = (rate / 5.0)**2
    decay_rate = (5.0 / max(1.0, word_expected_duration)) * urgency_factor
    floor_kappa = base_kappa * 0.8
    val = floor_kappa + (0.99 - floor_kappa) / (1 + np.exp(decay_rate * (t_elapsed - word_expected_duration / 2.5)))
    return val

def get_dynamic_logT(rate, gammas, dwell_times):
    """
    Computes the log transition matrix for the current time step.
    """
    n = len(STATES)
    logT = np.full((n, n), -np.inf)

    expected_durations = {}
    for s_from in STATES:
        if s_from in EMIT_STATES:
            base_kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            expected_durations[s_from] = 1.0 / (1.0 - base_kappa) if (1.0 - base_kappa) > 0 else 1.0
        else:
            expected_durations[s_from] = 1.0

    for s_from_idx in range(n):
        s_from = STATES[s_from_idx]
        row = pi_bar[s_from_idx].copy()

        if s_from in {"START", "STOP"}:
            logT[s_from_idx, :] = np.log(row + 1e-12)
        else:
            t_elapsed = dwell_times[s_from_idx]
            base_kappa = kappa_from_gamma_rate(gammas[s_from], rate)
            kappa_t = dynamic_kappa_refined(t_elapsed, base_kappa, expected_durations[s_from], rate)

            row = (1 - kappa_t) * row
            row[s_from_idx] += kappa_t
            logT[s_from_idx, :] = np.log(row / row.sum() + 1e-12)

    return logT


## variable inferred by the viterbi alogorithem

## variable inferred by the viterbi algorithm/ selecting the most prob variable
def viterbi_decode_dynamic(a_data, rate, gammas):
    T_len = len(a_data)
    n = len(STATES)
    V = np.full((T_len, n), -np.inf)
    B = np.zeros((T_len, n), dtype=int)
    D = np.zeros((T_len, n), dtype=int)
    logE = emission_log_probs(a_data)

    start_row = np.log(pi_bar[i["START"]] + 1e-12)
    for s_idx in range(n):
        if STATES[s_idx] in EMIT_STATES:
            V[0, s_idx] = start_row[s_idx] + logE[0, s_idx]
            D[0, s_idx] = 1

    for t in range(1, T_len):
        logT_t = get_dynamic_logT(rate, gammas, D[t-1])
        for j in range(n):
            prev_path_probs = V[t-1, :] + logT_t[:, j]
            best_prev_k = np.argmax(prev_path_probs)

            V[t, j] = prev_path_probs[best_prev_k] + logE[t, j]
            B[t, j] = best_prev_k

            if j == best_prev_k:
                D[t, j] = D[t-1, best_prev_k] + 1
            else:
                D[t, j] = 1

    path = np.zeros(T_len, dtype=int)
    path[-1] = np.argmax(V[-1])
    for t in range(T_len - 2, -1, -1):
        path[t] = B[t + 1, path[t + 1]]

    return [STATES[k] for k in path]

# Helper functions
def emission_log_probs(a_data):
    T = len(a_data)
    n = len(STATES)
    logp = np.full((T, n), -np.inf)
    for t in range(T):
        for s in EMIT_STATES:
            logp[t, i[s]] = norm.logpdf(a_data[t, 0], loc=mu[s], scale=sigma[s])
    return logp

def compress(seq):
    return [s for idx, s in enumerate(seq) if idx == 0 or s != seq[idx-1]]

def summarize(name, seqs):
    c = Counter(tuple(s) for s in seqs)
    total = sum(c.values())
    print(f"\n{name} (N={total}) top 8:")
    for (s, cnt) in c.most_common(8):
        print(f"  {list(s)!s:<40} | {cnt:4d} | {100*cnt/total:5.1f}%")
    has_or = sum(1 for s in seqs if "or" in s)
    print(f"  -> 'or' present: {has_or}/{total} = {100*has_or/total:4.1f}%")

# =================================================================
# == NEW EXPERIMENT: ISOLATING THE PERCEPTUAL EFFECT ==
# =================================================================

def run_isolated_perception_experiment(n_trials=1000, rate_slow=4.0, rate_fast=40.0):
    rng = np.random.default_rng(1234)
    gammas = sample_gammas(seed=2025)
    seq_slow_perception = []
    seq_fast_perception = []

    print("\n\n=== Isolating the Perceptual Effect of Rate ===")

    for k in range(n_trials):
        if k % 100 == 0: print(f"Trial {k}/{n_trials}...")

        # 1. Generate ONE ambiguous utterance at an intermediate rate.
        rate_generative = 5.5
        _, ambiguous_acoustic_signal = simulate_one(rate_generative, gammas, rng=rng)

        if len(ambiguous_acoustic_signal) < 3: continue

        # 2. Decode this SAME signal with a "slow-rate brain" model.
        zhat_slow = viterbi_decode_dynamic(ambiguous_acoustic_signal, rate_slow, gammas)
        seq_slow_perception.append(compress(zhat_slow))

        # 3. Decode this SAME signal with a "fast-rate brain" model.
        zhat_fast = viterbi_decode_dynamic(ambiguous_acoustic_signal, rate_fast, gammas)
        seq_fast_perception.append(compress(zhat_fast))

    # 4. Summarize and compare the perceptual results
    summarize(f"PERCEPTION at SLOW r={rate_slow}", seq_slow_perception)
    summarize(f"PERCEPTION at FAST r={rate_fast}", seq_fast_perception)

# --- RUN THE SIMULATION ---
run_isolated_perception_experiment(n_trials=1000)



=== Isolating the Perceptual Effect of Rate ===
Trial 0/1000...


/tmp/ipython-input-3985358260.py:119: RuntimeWarning: overflow encountered in exp
  val = floor_kappa + (0.99 - floor_kappa) / (1 + np.exp(decay_rate * (t_elapsed - word_expected_duration / 2.5)))


Trial 100/1000...
Trial 200/1000...
Trial 300/1000...
Trial 400/1000...
Trial 500/1000...
Trial 600/1000...
Trial 700/1000...
Trial 800/1000...
Trial 900/1000...

PERCEPTION at SLOW r=4.0 (N=968) top 8:
  ['lei', 'sure', 'time']                  |  422 |  43.6%
  ['lei', 'sure', 'or', 'time']            |  332 |  34.3%
  ['lei', 'sure']                          |  157 |  16.2%
  ['lei', 'sure', 'or']                    |   14 |   1.4%
  ['or', 'time']                           |    7 |   0.7%
  ['lei', 'sure', 'lei', 'sure', 'or', 'time'] |    7 |   0.7%
  ['lei']                                  |    6 |   0.6%
  ['lei', 'time']                          |    4 |   0.4%
  -> 'or' present: 367/968 = 37.9%

PERCEPTION at FAST r=40.0 (N=968) top 8:
  ['lei', 'sure', 'or', 'time']            |  636 |  65.7%
  ['lei', 'sure', 'time']                  |  149 |  15.4%
  ['lei', 'sure']                          |  129 |  13.3%
  ['lei', 'sure', 'lei', 'sure', 'or', 'time'] |    9 |   0.9%
  ['

# **9/17 Revision\**
* How to choose T0?
* Vlterbi algorithm
* HMM speechrate paper
* plot the variables

